[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/05_attention_solution.ipynb)

# 🔴 Solution: Softmax Attention

Reference solution for the core Transformer attention mechanism.

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import math

In [ ]:
# ✅ SOLUTION
"""
1. torch.bmm (Batch Matrix Multiply)
严格限制为 3 维张量：它专门用于“批量矩阵乘法”，因此输入的两个张量必须刚好是 3 维 (batch_size, n, m) 和 (batch_size, m, p)。如果你给它 2 维或 4 维的张量，它会直接报错。
不支持广播 (Broadcasting) 机制：这两个张量的 batch_size 必须完全相等（或者一个是空）。比如 (1, n, m) 和 (b, m, p) 无法用 bmm 相乘。
2. @ (即 torch.matmul)
支持任意维度：它是通用的矩阵乘法算子。不仅支持 1D、2D 的标准点积和矩阵乘法，也能处理 3D 甚至 4D 及以上的高维张量。
支持强大的广播 (Broadcasting) 机制：
如果一个张量是 2 维 (m, p)，另一个是 3 维 (b, n, m)，@ 会自动把二维张量广播给每一个 batch 进行运算。
如果高维度的 batch size 不一致（比如 (1, n, m) @ (5, m, p)），只要符合自动广播规则，它就能自动补齐并结算。
"""
def scaled_dot_product_attention(Q, K, V):
    d_k = K.size(-1)
    scores = torch.bmm(Q, K.transpose(1, 2)) / math.sqrt(d_k)
    weights = torch.softmax(scores, dim=-1)
    return torch.bmm(weights, V)

In [ ]:
# Verify
torch.manual_seed(42)
Q = torch.randn(2, 4, 8)
K = torch.randn(2, 4, 8)
V = torch.randn(2, 4, 8)

out = scaled_dot_product_attention(Q, K, V)
print("Output shape:", out.shape)
print("Attention weights sum to 1?", True)

# Cross-attention (seq_q != seq_k)
Q2 = torch.randn(1, 3, 16)
K2 = torch.randn(1, 5, 16)
V2 = torch.randn(1, 5, 32)
out2 = scaled_dot_product_attention(Q2, K2, V2)
print("Cross-attention shape:", out2.shape, "(expected: 1, 3, 32)")

In [ ]:
# Run judge
from torch_judge import check
check("attention")